# Reçete Okuyucu — Colab'da L4 testi

Bu notebook, reçete okuyucuyu sunucudaki kurulumla aynı GPU'da (NVIDIA L4, 24 GB) dener:

1. vLLM ile `Qwen/Qwen3-VL-8B-Instruct` modelini başlatır (Docker yerine doğrudan süreç olarak).
2. İki örnek reçeteyle (spec Ek A) entegrasyon testlerini çalıştırır.
3. Okuyucu servisini başlatır; kendi görselinizi yükleyip JSON yanıtı görebilirsiniz.

**Gereksinim:** *Çalışma zamanı → Çalışma zamanı türünü değiştir → **L4 GPU*** (Colab Pro). T4'te 8B model sığmaz ve bf16 desteklenmez.

**Süre:** Kurulum ~5–10 dk, model indirme (~17 GB) ve açılış ~5–10 dk.

**KVKK:** Colab'a yüklenen her görsel Google'ın bulutuna gider. Burada yalnızca kişisel bilgileri karartılmış örnekler kullanın; gerçek doğrulanmış reçetelerle toplu değerlendirmeyi kendi sunucunuzda yapın.

**Özel repo ise:** Colab'ın sol menüsündeki 🔑 *Secrets* bölümüne `GITHUB_TOKEN` adıyla, bu repoyu okuma yetkisi olan bir GitHub token'ı ekleyin ve notebook erişimini açın.

In [ ]:
# 1) GPU kontrolü
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(out.stdout or out.stderr)
if "L4" not in out.stdout:
    print("UYARI: L4 GPU seçili değil. Sonuçlar ve süreler sunucuyu yansıtmayabilir; T4'te model sığmaz.")

## 2) Repoyu klonla

In [ ]:
import os, subprocess

REPO = "FatihBrck/recete"
BRANCH = "feat/recete-okuyucu"  # Pull Request birleştirildikten sonra "main" yapın

token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass  # secret yok veya erişim verilmedi -> herkese açık repo varsayılır

if not os.path.isdir("/content/recete"):
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/content/recete"], check=True)
    # token'ı .git/config içinde bırakma
    subprocess.run(["git", "-C", "/content/recete", "remote", "set-url", "origin",
                    f"https://github.com/{REPO}.git"], check=True)

%cd /content/recete
!git log --oneline -1

## 3) Kurulum

vLLM, Colab'ın kendi PyTorch kurulumuyla çakışmasın diye ayrı bir sanal ortama kurulur (sürüm `docker-compose.yml` ile aynı). Okuyucu servisi Colab'ın Python'una kurulur.

In [ ]:
!pip install -q uv
!uv venv -q /content/vllm-env --python 3.12
!uv pip install -q --python /content/vllm-env/bin/python "vllm==0.34.0"
!pip install -q -e ".[dev]"

## 4) vLLM'i başlat

Ayarlar `docker-compose.yml` ile aynıdır. İlk çalıştırmada model indirilir; hücre sunucu hazır olana kadar bekler (en fazla 30 dk). Hata olursa logun son kısmı yazdırılır.

In [ ]:
import subprocess, time, urllib.request

def wait_for(url, proc, timeout_s, log_path):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            print(open(log_path).read()[-4000:])
            raise RuntimeError(f"süreç beklenmedik şekilde kapandı (kod {proc.returncode})")
        try:
            urllib.request.urlopen(url, timeout=5)
            print(f"hazır: {url} ({time.time() - t0:.0f} sn)")
            return
        except Exception:
            time.sleep(10)
    print(open(log_path).read()[-4000:])
    raise TimeoutError(url)

VLLM_LOG = "/content/vllm.log"
vllm_cmd = [
    "/content/vllm-env/bin/vllm", "serve", "Qwen/Qwen3-VL-8B-Instruct",
    "--dtype", "bfloat16",
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.92",
    "--max-num-seqs", "4",
    "--limit-mm-per-prompt", '{"image":1,"video":0}',
    "--port", "8000",
]
vllm_proc = subprocess.Popen(vllm_cmd, stdout=open(VLLM_LOG, "w"), stderr=subprocess.STDOUT)
wait_for("http://localhost:8000/v1/models", vllm_proc, 1800, VLLM_LOG)

## 5) Örnek reçetelerle entegrasyon testi

`tests/integration/test_golden.py`: el yazısı (A.1) ve basılı form (A.2) örnekleri gerçek modelle okunur; değerler, güven puanının hesaplanabildiği ve sağ göz SPH güveninin > 0.5 olduğu kontrol edilir.

In [ ]:
!READER_VLLM_URL=http://localhost:8000/v1 python -m pytest -m integration -v

## 6) Okuyucu servisini başlat

`READER_VLLM_BASE_URL`, `config/models.yaml` içindeki Docker adresini (`http://vllm:8000/v1`) Colab'daki yerel adresle değiştirir.

In [ ]:
import os, subprocess

API_LOG = "/content/reader.log"
api_env = dict(os.environ, READER_VLLM_BASE_URL="http://localhost:8000/v1")
api_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "--factory", "reader.api:create_app", "--host", "127.0.0.1", "--port", "8080"],
    cwd="/content/recete", env=api_env, stdout=open(API_LOG, "w"), stderr=subprocess.STDOUT,
)
wait_for("http://127.0.0.1:8080/health", api_proc, 120, API_LOG)

## 7) Örnek bir reçeteyi API üzerinden oku

In [ ]:
import json, httpx

def read_image(name, data):
    r = httpx.post("http://127.0.0.1:8080/v1/read", files={"file": (name, data)}, timeout=120)
    print(name, "->", r.status_code)
    print(json.dumps(r.json(), ensure_ascii=False, indent=2))

path = "tests/fixtures/golden/a1_handwritten.webp"
read_image(path, open(path, "rb").read())

## 8) Kendi görselinizi deneyin

JPEG, PNG, WEBP veya HEIC, en fazla 15 MB. **Kişisel bilgileri karartılmış** görseller kullanın (KVKK).

In [ ]:
from google.colab import files

uploaded = files.upload()
for name, data in uploaded.items():
    read_image(name, data)

## 9) (İsteğe bağlı) Küçük bir değerlendirme

Manifest biçimi README'nin *Değerlendirme* bölümünde. Görselleri ve `manifest.jsonl` dosyasını `/content/data/` altına yükleyip aşağıdaki satırların başındaki `#` işaretini kaldırın. Asıl test setini (tür başına ~100 reçete) kendi sunucunuzda çalıştırın.

In [ ]:
# !python -m eval.run_eval run --manifest /content/data/manifest.jsonl --url http://127.0.0.1:8080 --out /content/preds.jsonl
# !python -m eval.run_eval report --manifest /content/data/manifest.jsonl --predictions /content/preds.jsonl --out /content/report
# print(open("/content/report/report.md").read())

## 10) Kapat

In [ ]:
api_proc.terminate()
vllm_proc.terminate()
print("Servisler durduruldu. Colab oturumu kapanınca indirilen model de silinir.")